# Section 1: Load Deutsche Bahn Monthly Data & Filter ICE

## Objective

Download **one monthly Parquet file** from Hugging Face, filter `train_type == "ICE"`, inspect the result, and save it to disk for all later notebooks.

## Why this step is important

Notebook 01 only used 100 preview rows. This section loads **real monthly data** — the first actual acquisition step.

## What problem does it solve?

It produces a saved ICE-only dataset file so Notebooks 03–10 can `read_parquet()` from disk without re-downloading.

## Methodology

1. Load config from `data/reference/project_config.json`.
2. Download `monthly_processed_data/data-2024-07.parquet` via `huggingface_hub` (cached locally after first run).
3. Filter `train_type == "ICE"` while reading (chunked — avoids loading all train types into RAM).
4. Verify 17 columns match the documented schema.
5. Save `data/processed/ice_raw_2024-07.parquet` + load report JSON.

**Why one month?** Full dataset = 54.8 GB. One month is manageable; more months can be added later.

**Why chunked read?** A monthly file contains all train types (S, RE, ICE…). Chunked filtering keeps memory usage low.

## Expected Output

- Download progress (first run may take several minutes)
- ICE row count for July 2024
- Column + delay summary
- Saved parquet file on disk

## Interpretation

- **ICE rows << total rows** → normal; ICE is a small share of all German rail traffic
- **All 17 columns present** → schema matches Notebook 01
- **`delay_in_min` negative values** → early arrivals; expected
- **File saved** → Notebook 03 loads from `ice_raw_2024-07.parquet`, not Hugging Face

## Challenges Faced

| Challenge | Approach |
|-----------|----------|
| 54.8 GB full dataset | One month only |
| Mixed train types in file | Filter ICE during chunked read |
| Large download | `huggingface_hub` caches file locally |

## Summary

Real ICE data for July 2024 is downloaded, filtered, inspected, and saved to disk.

## Next Step

**Section 2:** Deep inspection — null rates, delay distribution, cancellation rate, unique stations/trains.

---

**Key Takeaways**
- First real data load; ICE filter applied
- Output: `data/processed/ice_raw_2024-07.parquet`
- Later notebooks load from this file

**What Comes Next**
- Section 2: detailed profiling of the saved ICE dataset

In [2]:
# =============================================================================
# Notebook 02 | Section 1: Load Deutsche Bahn Monthly Data & Filter ICE
# =============================================================================
# Self-contained: loads config from disk; downloads ONE monthly Parquet;
# filters ICE; saves to data/processed/ for later notebooks.
# =============================================================================

from __future__ import annotations

import json
import sys
import time
from datetime import datetime, timezone
from pathlib import Path

import pandas as pd
import pyarrow.parquet as pq

# =============================================================================
# 1. PATHS & SETTINGS
# =============================================================================
PROJECT_ROOT = Path.cwd()
REFERENCE_DIR = PROJECT_ROOT / "data" / "reference"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"

CONFIG_PATH = REFERENCE_DIR / "project_config.json"

# Which month to load (change here to process a different month later)
TARGET_MONTH = "2024-07"          # format: YYYY-MM
ICE_FILTER = "ICE"
CHUNK_SIZE = 100_000              # rows per chunk during filtered read

OUTPUT_PARQUET = PROCESSED_DIR / f"ice_raw_{TARGET_MONTH}.parquet"
LOAD_REPORT_PATH = REFERENCE_DIR / f"ice_load_report_{TARGET_MONTH}.json"

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
REFERENCE_DIR.mkdir(parents=True, exist_ok=True)


def load_json(path: Path) -> dict:
    """Load JSON config from disk."""
    if not path.exists():
        raise FileNotFoundError(
            f"Missing: {path}\nRun Notebook 01 first."
        )
    with path.open("r", encoding="utf-8") as f:
        return json.load(f)


config = load_json(CONFIG_PATH)

HF_DATASET_ID = config["data_sources"]["deutsche_bahn"]["hf_dataset_id"]
VERIFIED_COLUMNS = [c["name"] for c in config["db_verified_columns"]]
PARQUET_FILENAME = (
    config["data_sources"]["deutsche_bahn"]["monthly_parquet_pattern"]
    .format(year_month=TARGET_MONTH)
)

# =============================================================================
# 2. PACKAGE CHECK
# =============================================================================
def require_package(import_name: str, pip_name: str) -> None:
    """Raise clear error if a required package is missing."""
    try:
        __import__(import_name)
    except ImportError:
        raise ImportError(
            f"Package '{import_name}' not installed.\n"
            f"Run: pip install {pip_name}"
        )


require_package("huggingface_hub", "huggingface_hub")
from huggingface_hub import hf_hub_download

# =============================================================================
# 3. DOWNLOAD MONTHLY PARQUET (cached after first download)
# =============================================================================
print(f"Downloading: {HF_DATASET_ID}/{PARQUET_FILENAME}")
print("(First run may take several minutes — file is cached locally after that.)")
print()

download_start = time.time()

local_parquet_path = hf_hub_download(
    repo_id=HF_DATASET_ID,
    filename=PARQUET_FILENAME,
    repo_type="dataset",
)

download_seconds = round(time.time() - download_start, 1)
print(f"Download/cache complete in {download_seconds}s")
print(f"Local path: {local_parquet_path}")
print()

# =============================================================================
# 4. CHUNKED ICE FILTER (memory-safe — does not load full month at once)
# =============================================================================
def load_ice_chunked(
    parquet_path: str,
    columns: list[str],
    train_type: str = ICE_FILTER,
    chunk_size: int = CHUNK_SIZE,
) -> tuple[pd.DataFrame, int]:
    """
    Read Parquet in chunks and keep only ICE rows.

    Returns
    -------
    ice_df : filtered ICE DataFrame
    total_rows_scanned : total rows read across all chunks
    """
    parquet_file = pq.ParquetFile(parquet_path)
    ice_chunks: list[pd.DataFrame] = []
    total_rows_scanned = 0

    for batch in parquet_file.iter_batches(batch_size=chunk_size, columns=columns):
        chunk = batch.to_pandas()
        total_rows_scanned += len(chunk)
        ice_chunk = chunk[chunk["train_type"] == train_type]
        if not ice_chunk.empty:
            ice_chunks.append(ice_chunk)

        # Progress every ~500k rows
        if total_rows_scanned % 500_000 < chunk_size:
            ice_so_far = sum(len(c) for c in ice_chunks)
            print(f"  Scanned {total_rows_scanned:>10,} rows | ICE found: {ice_so_far:>8,}")

    if not ice_chunks:
        raise ValueError(
            f"No ICE rows found in {parquet_path}. "
            f"Check TARGET_MONTH='{TARGET_MONTH}' and train_type filter."
        )

    ice_df = pd.concat(ice_chunks, ignore_index=True)
    return ice_df, total_rows_scanned


print(f"Filtering train_type == '{ICE_FILTER}' (chunked read)...")
filter_start = time.time()

ice_df, total_rows_scanned = load_ice_chunked(
    parquet_path=local_parquet_path,
    columns=VERIFIED_COLUMNS,
)

filter_seconds = round(time.time() - filter_start, 1)
ice_pct = round(100 * len(ice_df) / total_rows_scanned, 2)

print()
print(f"Filtering complete in {filter_seconds}s")
print(f"  Total rows scanned : {total_rows_scanned:,}")
print(f"  ICE rows kept      : {len(ice_df):,}  ({ice_pct}% of month)")
print()

# =============================================================================
# 5. SCHEMA VALIDATION
# =============================================================================
missing_cols = [c for c in VERIFIED_COLUMNS if c not in ice_df.columns]
extra_cols = [c for c in ice_df.columns if c not in VERIFIED_COLUMNS]

if missing_cols:
    raise ValueError(f"Missing expected columns: {missing_cols}")
if extra_cols:
    raise ValueError(f"Unexpected extra columns: {extra_cols}")

# Confirm only ICE train type remains
non_ice = ice_df[ice_df["train_type"] != ICE_FILTER]
if len(non_ice) > 0:
    raise ValueError(f"Filter failed: {len(non_ice)} non-ICE rows remain.")

# =============================================================================
# 6. QUICK INSPECTION
# =============================================================================
# Parse timestamps for summary stats
timestamp_cols = [
    "time",
    "arrival_planned_time",
    "arrival_change_time",
    "departure_planned_time",
    "departure_change_time",
]
for col in timestamp_cols:
    ice_df[col] = pd.to_datetime(ice_df[col], errors="coerce")

delay_stats = {
    "min": int(ice_df["delay_in_min"].min()),
    "max": int(ice_df["delay_in_min"].max()),
    "mean": round(float(ice_df["delay_in_min"].mean()), 2),
    "median": float(ice_df["delay_in_min"].median()),
    "early_count": int((ice_df["delay_in_min"] < 0).sum()),
    "on_time_count": int((ice_df["delay_in_min"] == 0).sum()),
    "late_count": int((ice_df["delay_in_min"] > 0).sum()),
}

inspection_summary = {
    "rows": len(ice_df),
    "columns": len(ice_df.columns),
    "unique_stations_eva": int(ice_df["eva"].nunique()),
    "unique_train_numbers": int(ice_df["train_number"].nunique()),
    "unique_rides": int(ice_df["train_line_ride_id"].nunique()),
    "canceled_stops": int(ice_df["is_canceled"].sum()),
    "canceled_pct": round(100 * ice_df["is_canceled"].mean(), 2),
    "date_range_time_col": {
        "min": str(ice_df["time"].min()),
        "max": str(ice_df["time"].max()),
    },
    "delay_stats": delay_stats,
}

# =============================================================================
# 7. SAVE TO DISK (later notebooks load this file — not Hugging Face)
# =============================================================================
ice_df.to_parquet(OUTPUT_PARQUET, index=False)

load_report = {
    "metadata": {
        "notebook": "02_Data_Acquisition_and_Initial_Inspection",
        "section": "Section 1",
        "created_at_utc": datetime.now(timezone.utc).isoformat(),
        "target_month": TARGET_MONTH,
        "hf_dataset_id": HF_DATASET_ID,
        "source_file": PARQUET_FILENAME,
        "local_parquet_cache": local_parquet_path,
    },
    "processing": {
        "ice_filter": ICE_FILTER,
        "chunk_size": CHUNK_SIZE,
        "total_rows_scanned": total_rows_scanned,
        "ice_rows_saved": len(ice_df),
        "ice_pct_of_month": ice_pct,
        "download_seconds": download_seconds,
        "filter_seconds": filter_seconds,
    },
    "output_files": {
        "ice_parquet": str(OUTPUT_PARQUET),
        "load_report": str(LOAD_REPORT_PATH),
    },
    "inspection_summary": inspection_summary,
    "columns": VERIFIED_COLUMNS,
}

with LOAD_REPORT_PATH.open("w", encoding="utf-8") as f:
    json.dump(load_report, f, indent=2, ensure_ascii=False)

# =============================================================================
# 8. PRINT SUMMARY
# =============================================================================
sep = "=" * 72
print(sep)
print("Section 1: Load Deutsche Bahn Monthly Data & Filter ICE")
print(sep)
print(f"Month          : {TARGET_MONTH}")
print(f"Source         : {HF_DATASET_ID}")
print(f"Rows scanned   : {total_rows_scanned:,}")
print(f"ICE rows saved : {len(ice_df):,}  ({ice_pct}%)")
print()

print("--- INSPECTION SUMMARY ---")
for key, val in inspection_summary.items():
    if key != "delay_stats":
        print(f"  {key}: {val}")
print(f"  delay_stats: {delay_stats}")
print()

print("--- SAMPLE ICE ROWS (first 3) ---")
print(ice_df.head(3).to_string(index=False))
print()

print("--- SAVED ---")
print(f"  ICE data   : {OUTPUT_PARQUET}")
print(f"  Load report: {LOAD_REPORT_PATH}")
print()
print("Later notebooks load with:")
print(f'  pd.read_parquet("{OUTPUT_PARQUET}")')
print()
print("Ready for Section 2: detailed ICE data profiling.")
print(sep)

Downloading: piebro/deutsche-bahn-data/monthly_processed_data/data-2024-07.parquet
(First run may take several minutes — file is cached locally after that.)



monthly_processed_data/data-2024-07.parq(…):   0%|          | 0.00/107M [00:00<?, ?B/s]

c:\Users\Manikanta\anaconda3\Lib\site-packages\huggingface_hub\file_download.py:137: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Manikanta\.cache\huggingface\hub\datasets--piebro--deutsche-bahn-data. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


Download/cache complete in 24.5s
Local path: C:\Users\Manikanta\.cache\huggingface\hub\datasets--piebro--deutsche-bahn-data\snapshots\4b211417bcefd8cd4150d68b889c9411162a1ede\monthly_processed_data\data-2024-07.parquet

Filtering train_type == 'ICE' (chunked read)...
  Scanned    500,000 rows | ICE found:   41,897
  Scanned  1,000,000 rows | ICE found:   83,806
  Scanned  1,500,000 rows | ICE found:  121,717
  Scanned  2,000,000 rows | ICE found:  157,416
  Scanned  2,007,251 rows | ICE found:  157,886

Filtering complete in 2.4s
  Total rows scanned : 2,007,251
  ICE rows kept      : 157,886  (7.87% of month)

Section 1: Load Deutsche Bahn Monthly Data & Filter ICE
Month          : 2024-07
Source         : piebro/deutsche-bahn-data
Rows scanned   : 2,007,251
ICE rows saved : 157,886  (7.87%)

--- INSPECTION SUMMARY ---
  rows: 157886
  columns: 17
  unique_stations_eva: 95
  unique_train_numbers: 812
  unique_rides: 1065
  canceled_stops: 11497
  canceled_pct: 7.28
  date_range_time_c

# Section 2: ICE Dataset Profiling & Quality Assessment

## Objective

Load the saved ICE Parquet from Section 1 and profile it: null rates, delay distribution, cancellations, stations, and trains. Save the report to disk.

## Why this step is important

Before cleaning or merging, we must understand data quality — missing values, delay patterns, and how many canceled stops exist.

## What problem does it solve?

It answers: *Is this ICE dataset complete enough to clean and merge with weather?*

## Methodology

1. Auto-find project root and load `ice_raw_2024-07.parquet` from disk.
2. Profile every column: dtype, null %, unique values.
3. Analyze `delay_in_min`: early / on-time / late counts.
4. Check `is_canceled` rate and `train_type` (should be 100% ICE).
5. Summarize unique `eva` stations and `train_number` services.
6. Save `data/reference/ice_profile_report_2024-07.json`.

## Expected Output

- Column profile table sorted by null %
- Delay distribution summary
- Cancellation rate
- Unique station/train counts
- Saved JSON profile report

## Interpretation

- **High null on `station_name` or `line_number`** → normal for ICE; use `eva` / `xml_station_name`
- **Most delays = 0** → class imbalance expected for classification
- **Canceled stops** → decide exclusion policy in Notebook 03
- **Many unique `eva`** → geocoding workload for Notebook 04

## Challenges Faced

| Challenge | Approach |
|-----------|----------|
| Large ICE file | Load from saved parquet, not Hugging Face |
| Path mismatch | `find_project_root()` auto-detects `data/` folder |

## Summary

ICE data for July 2024 is fully profiled. Quality issues are documented before cleaning.

## Next Step

**Section 3:** Inspect Open-Meteo weather sample for the same date range and compare time format with railway timestamps.

---

**Key Takeaways**
- Profiling happens on saved ICE file, not raw HF download
- Delay and cancellation stats guide Notebook 03 cleaning rules

**What Comes Next**
- Section 3: weather data inspection aligned to July 2024

In [3]:
# =============================================================================
# Notebook 02 | Section 2: ICE Dataset Profiling & Quality Assessment
# =============================================================================
# Self-contained: loads ICE parquet + config from disk.
# No dependency on Section 1 kernel state.
# =============================================================================

from __future__ import annotations

import json
from datetime import datetime, timezone
from pathlib import Path

import pandas as pd

# =============================================================================
# 1. PATHS — auto-find project root
# =============================================================================
def find_project_root() -> Path:
    """Find folder containing data/reference/project_config.json."""
    for candidate in [Path.cwd(), *Path.cwd().parents]:
        config_file = candidate / "data" / "reference" / "project_config.json"
        if config_file.exists():
            return candidate
    raise FileNotFoundError(
        "Cannot find data/reference/project_config.json.\n"
        "Run Notebook 01 and Notebook 02 Section 1 first."
    )


PROJECT_ROOT = find_project_root()
REFERENCE_DIR = PROJECT_ROOT / "data" / "reference"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"

CONFIG_PATH = REFERENCE_DIR / "project_config.json"
TARGET_MONTH = "2024-07"
ICE_PARQUET = PROCESSED_DIR / f"ice_raw_{TARGET_MONTH}.parquet"
PROFILE_REPORT_PATH = REFERENCE_DIR / f"ice_profile_report_{TARGET_MONTH}.json"
LOAD_REPORT_PATH = REFERENCE_DIR / f"ice_load_report_{TARGET_MONTH}.json"


def load_json(path: Path) -> dict:
    if not path.exists():
        raise FileNotFoundError(f"Missing: {path}")
    with path.open("r", encoding="utf-8") as f:
        return json.load(f)


# =============================================================================
# 2. LOAD CONFIG + ICE DATA FROM DISK
# =============================================================================
config = load_json(CONFIG_PATH)
VERIFIED_COLUMNS = [c["name"] for c in config["db_verified_columns"]]
ICE_FILTER = config["scope"]["train_type_filter"]

if not ICE_PARQUET.exists():
    raise FileNotFoundError(
        f"Missing: {ICE_PARQUET}\n"
        "Run Notebook 02 Section 1 first to create the ICE parquet file."
    )

print(f"Loading ICE data from: {ICE_PARQUET}")
ice_df = pd.read_parquet(ICE_PARQUET)

# Optional: cross-check with Section 1 load report if available
if LOAD_REPORT_PATH.exists():
    load_report = load_json(LOAD_REPORT_PATH)
    expected_rows = load_report["processing"]["ice_rows_saved"]
    if len(ice_df) != expected_rows:
        print(
            f"WARNING: Row count mismatch. "
            f"Section 1 saved {expected_rows:,}, now loaded {len(ice_df):,}"
        )
else:
    print("Note: load report not found — skipping row count cross-check.")

# =============================================================================
# 3. SCHEMA CHECK
# =============================================================================
missing_cols = [c for c in VERIFIED_COLUMNS if c not in ice_df.columns]
if missing_cols:
    raise ValueError(f"Missing columns in saved ICE file: {missing_cols}")

ice_df = ice_df[VERIFIED_COLUMNS]  # enforce column order

non_ice = ice_df[ice_df["train_type"] != ICE_FILTER]
if len(non_ice) > 0:
    raise ValueError(f"Expected only ICE rows, found {len(non_ice)} non-ICE rows.")

# =============================================================================
# 4. PARSE TIMESTAMPS
# =============================================================================
timestamp_cols = [
    "time",
    "arrival_planned_time",
    "arrival_change_time",
    "departure_planned_time",
    "departure_change_time",
]
for col in timestamp_cols:
    ice_df[col] = pd.to_datetime(ice_df[col], errors="coerce")

# =============================================================================
# 5. COLUMN PROFILING
# =============================================================================
def profile_columns(df: pd.DataFrame) -> pd.DataFrame:
    """Return per-column dtype, null count, null %, and unique values."""
    rows = []
    for col in df.columns:
        rows.append({
            "column": col,
            "dtype": str(df[col].dtype),
            "null_count": int(df[col].isna().sum()),
            "null_pct": round(100 * df[col].isna().mean(), 2),
            "n_unique": int(df[col].nunique(dropna=True)),
        })
    return pd.DataFrame(rows).sort_values("null_pct", ascending=False).reset_index(drop=True)


column_profile = profile_columns(ice_df)

# =============================================================================
# 6. DELAY DISTRIBUTION
# =============================================================================
delay = ice_df["delay_in_min"]

delay_distribution = {
    "count": int(len(delay)),
    "min": int(delay.min()),
    "max": int(delay.max()),
    "mean": round(float(delay.mean()), 2),
    "median": float(delay.median()),
    "std": round(float(delay.std()), 2),
    "early_arrivals_lt_0": int((delay < 0).sum()),
    "on_time_eq_0": int((delay == 0).sum()),
    "late_gt_0": int((delay > 0).sum()),
    "late_gte_6": int((delay >= 6).sum()),   # matches classification threshold
    "late_gte_15": int((delay >= 15).sum()),
}

# Top delay values
delay_value_counts = (
    delay.value_counts()
    .head(10)
    .rename_axis("delay_in_min")
    .reset_index(name="count")
)

# =============================================================================
# 7. CANCELLATION & STATION / TRAIN SUMMARY
# =============================================================================
canceled_count = int(ice_df["is_canceled"].sum())
canceled_pct = round(100 * ice_df["is_canceled"].mean(), 2)

station_summary = {
    "unique_eva_stations": int(ice_df["eva"].nunique()),
    "unique_station_names": int(ice_df["station_name"].nunique(dropna=True)),
    "top_5_stations_by_stops": (
        ice_df["xml_station_name"]
        .value_counts()
        .head(5)
        .to_dict()
    ),
}

train_summary = {
    "unique_train_numbers": int(ice_df["train_number"].nunique()),
    "unique_rides": int(ice_df["train_line_ride_id"].nunique()),
    "top_5_trains_by_stops": (
        ice_df["train_number"]
        .value_counts()
        .head(5)
        .to_dict()
    ),
    "line_number_null_pct": round(100 * ice_df["line_number"].isna().mean(), 2),
}

time_range = {
    "time_min": str(ice_df["time"].min()),
    "time_max": str(ice_df["time"].max()),
    "departure_planned_min": str(ice_df["departure_planned_time"].min()),
    "departure_planned_max": str(ice_df["departure_planned_time"].max()),
}

# =============================================================================
# 8. DATA QUALITY FLAGS (for Notebook 03)
# =============================================================================
quality_flags = []

# Flag high-null columns (> 10%)
high_null = column_profile[column_profile["null_pct"] > 10]["column"].tolist()
if high_null:
    quality_flags.append({
        "flag": "high_null_columns",
        "columns": high_null,
        "action": "Document in Notebook 03; do not drop without justification",
    })

# Flag canceled stops
if canceled_pct > 0:
    quality_flags.append({
        "flag": "canceled_stops_present",
        "count": canceled_count,
        "pct": canceled_pct,
        "action": "Exclude is_canceled==True in Notebook 03 (per merge_strategy)",
    })

# Flag class imbalance for classification
on_time_pct = round(100 * delay_distribution["on_time_eq_0"] / len(delay), 2)
late_gte_6_pct = round(100 * delay_distribution["late_gte_6"] / len(delay), 2)
quality_flags.append({
    "flag": "class_imbalance_expected",
    "on_time_pct": on_time_pct,
    "delayed_gte_6_pct": late_gte_6_pct,
    "action": "Use F1/ROC-AUC in modeling (defined in research_framework.json)",
})

# =============================================================================
# 9. SAVE PROFILE REPORT
# =============================================================================
profile_report = {
    "metadata": {
        "notebook": "02_Data_Acquisition_and_Initial_Inspection",
        "section": "Section 2",
        "created_at_utc": datetime.now(timezone.utc).isoformat(),
        "target_month": TARGET_MONTH,
        "source_parquet": str(ICE_PARQUET),
    },
    "dataset_shape": {
        "rows": int(len(ice_df)),
        "columns": int(len(ice_df.columns)),
    },
    "column_profile": column_profile.to_dict(orient="records"),
    "delay_distribution": delay_distribution,
    "top_delay_values": delay_value_counts.to_dict(orient="records"),
    "cancellation": {
        "canceled_count": canceled_count,
        "canceled_pct": canceled_pct,
    },
    "station_summary": station_summary,
    "train_summary": train_summary,
    "time_range": time_range,
    "quality_flags": quality_flags,
}

with PROFILE_REPORT_PATH.open("w", encoding="utf-8") as f:
    json.dump(profile_report, f, indent=2, ensure_ascii=False)

# =============================================================================
# 10. PRINT SUMMARY
# =============================================================================
sep = "=" * 72
print(sep)
print("Section 2: ICE Dataset Profiling & Quality Assessment")
print(sep)
print(f"Source  : {ICE_PARQUET}")
print(f"Rows    : {len(ice_df):,}")
print(f"Columns : {len(ice_df.columns)}")
print(f"Month   : {TARGET_MONTH}")
print()

print("--- COLUMN PROFILE (sorted by null %) ---")
print(column_profile.to_string(index=False))
print()

print("--- DELAY DISTRIBUTION ---")
for key, val in delay_distribution.items():
    print(f"  {key}: {val}")
print()

print("--- CANCELLATIONS ---")
print(f"  Canceled stops: {canceled_count:,}  ({canceled_pct}%)")
print()

print("--- STATIONS & TRAINS ---")
print(f"  Unique EVA stations  : {station_summary['unique_eva_stations']}")
print(f"  Unique train numbers : {train_summary['unique_train_numbers']}")
print(f"  Unique rides         : {train_summary['unique_rides']}")
print(f"  line_number null %   : {train_summary['line_number_null_pct']}%")
print()

print("--- TIME RANGE ---")
for key, val in time_range.items():
    print(f"  {key}: {val}")
print()

print("--- QUALITY FLAGS (for Notebook 03) ---")
for flag in quality_flags:
    print(f"  • {flag['flag']}: {flag.get('action', '')}")
print()

print("--- TOP 5 DELAY VALUES ---")
print(delay_value_counts.to_string(index=False))
print()

print(f"--- SAVED ---")
print(f"  {PROFILE_REPORT_PATH}")
print()
print("Ready for Section 3: Open-Meteo weather inspection for July 2024.")
print(sep)

Loading ICE data from: c:\Users\Manikanta\Desktop\ML Project\Notebooks\data\processed\ice_raw_2024-07.parquet
Section 2: ICE Dataset Profiling & Quality Assessment
Source  : c:\Users\Manikanta\Desktop\ML Project\Notebooks\data\processed\ice_raw_2024-07.parquet
Rows    : 157,886
Columns : 17
Month   : 2024-07

--- COLUMN PROFILE (sorted by null %) ---
                   column          dtype  null_count  null_pct  n_unique
              line_number         object      157886    100.00         0
   departure_planned_time datetime64[ns]       15452      9.79     37610
    departure_change_time datetime64[ns]       15441      9.78     38066
     arrival_planned_time datetime64[ns]       15310      9.70     37765
      arrival_change_time datetime64[ns]       15307      9.69     38099
             station_name         object           0      0.00        91
               train_type         object           0      0.00         1
   train_line_station_num          int32           0      0.00 

# Section 3: Open-Meteo Weather Fetch & Time Alignment Check

## Objective

Fetch hourly weather for the same date range as our ICE data (July 2024), parse the API response, demonstrate hour-rounding on railway timestamps, and save a weather sample to disk.

## Why this step is important

Railway times are minute-level; weather is hourly. We must confirm the time formats align **before** the merge in Notebook 04.

## What problem does it solve?

It answers: *Does `departure_planned_time` round correctly to match Open-Meteo `hourly.time`?*

## Methodology

1. Load ICE parquet from disk → extract date range.
2. Call Open-Meteo Archive API for July 2024 (demo location: Berlin).
3. Parse JSON → DataFrame; save `weather_sample_2024-07.parquet`.
4. Create `departure_planned_hour` on sample ICE rows and compare formats.
5. Save `weather_inspection_report_2024-07.json`.

**Note:** Full per-station weather for all `eva` codes happens in Notebook 04 (after geocoding).

## Expected Output

- ~744 hourly weather rows (31 days × 24 hours)
- Time format comparison table
- Hour-rounding demo on real ICE `departure_planned_time` values
- Saved weather parquet + inspection report

## Interpretation

- **Weather `time`** = `2024-07-01T14:00` in `Europe/Berlin`
- **Railway hour key** = `departure_planned_time` floored to hour in same timezone
- **Different row counts** → normal; one weather row serves many trains at that location/hour
- **Demo uses Berlin coords only** → full station coverage comes in Notebook 04

## Challenges Faced

| Challenge | Approach |
|-----------|----------|
| Weather is per lat/lon, not per eva | Geocode all stations in Notebook 04 |
| Hourly vs minute timestamps | Floor railway time to hour |
| API download for full month | Cache parquet locally |

## Summary

Weather data for July 2024 is fetched, parsed, and time-aligned with railway timestamps. Merge key logic is verified on real ICE rows.

## Next Step

**Section 4 (Notebook 02 closing):** Summarize acquisition phase, list all saved files, confirm readiness for Notebook 03 (cleaning).

---

**Key Takeaways**
- Weather time = hourly; railway time must be rounded
- One weather file per station-month in Notebook 04
- Output: `weather_sample_2024-07.parquet`

**What Comes Next**
- Notebook 02 close-out, then Notebook 03: timestamp standardization & cleaning

In [4]:
# =============================================================================
# Notebook 02 | Section 3: Open-Meteo Weather Fetch & Time Alignment Check
# =============================================================================
# Self-contained: loads ICE data + configs from disk; fetches weather via API.
# Saves weather parquet + inspection report for Notebook 04.
# =============================================================================

from __future__ import annotations

import json
from datetime import datetime, timezone
from pathlib import Path

import pandas as pd
import requests

# =============================================================================
# 1. PATHS
# =============================================================================
def find_project_root() -> Path:
    for candidate in [Path.cwd(), *Path.cwd().parents]:
        config_file = candidate / "data" / "reference" / "project_config.json"
        if config_file.exists():
            return candidate
    raise FileNotFoundError("Cannot find data/reference/project_config.json")


PROJECT_ROOT = find_project_root()
REFERENCE_DIR = PROJECT_ROOT / "data" / "reference"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"

CONFIG_PATH = REFERENCE_DIR / "project_config.json"
WEATHER_DICT_PATH = REFERENCE_DIR / "weather_data_dictionary.json"
TARGET_MONTH = "2024-07"
ICE_PARQUET = PROCESSED_DIR / f"ice_raw_{TARGET_MONTH}.parquet"
WEATHER_PARQUET = PROCESSED_DIR / f"weather_sample_{TARGET_MONTH}.parquet"
WEATHER_REPORT_PATH = REFERENCE_DIR / f"weather_inspection_report_{TARGET_MONTH}.json"

OPEN_METEO_ARCHIVE_URL = "https://archive-api.open-meteo.com/v1/archive"
REQUEST_TIMEOUT = 60

# Demo coordinates (Berlin area) — per-station coords added in Notebook 04
DEMO_LATITUDE = 52.525
DEMO_LONGITUDE = 13.369
DEMO_TIMEZONE = "Europe/Berlin"

HOURLY_VARIABLES = [
    "temperature_2m",
    "precipitation",
    "rain",
    "snowfall",
    "windspeed_10m",
    "windgusts_10m",
    "weather_code",
    "visibility",
]


def load_json(path: Path) -> dict:
    if not path.exists():
        raise FileNotFoundError(f"Missing: {path}")
    with path.open("r", encoding="utf-8") as f:
        return json.load(f)


# =============================================================================
# 2. LOAD CONFIG + ICE DATA
# =============================================================================
config = load_json(CONFIG_PATH)
weather_dict = load_json(WEATHER_DICT_PATH)

if not ICE_PARQUET.exists():
    raise FileNotFoundError(
        f"Missing: {ICE_PARQUET}\nRun Notebook 02 Section 1 first."
    )

ice_df = pd.read_parquet(ICE_PARQUET)

# Parse railway timestamps
timestamp_cols = [
    "time",
    "arrival_planned_time",
    "arrival_change_time",
    "departure_planned_time",
    "departure_change_time",
]
for col in timestamp_cols:
    ice_df[col] = pd.to_datetime(ice_df[col], errors="coerce")

# Derive date range from ICE data for API request
ice_date_min = ice_df["time"].min().date()
ice_date_max = ice_df["time"].max().date()
start_date = str(ice_date_min)
end_date = str(ice_date_max)

print(f"ICE date range: {start_date} to {end_date}")
print(f"ICE rows      : {len(ice_df):,}")
print(f"Unique eva    : {ice_df['eva'].nunique()}  (geocoding in Notebook 04)")
print()

# =============================================================================
# 3. FETCH OPEN-METEO WEATHER FOR ICE DATE RANGE
# =============================================================================
def fetch_open_meteo_archive(
    latitude: float,
    longitude: float,
    start_date: str,
    end_date: str,
    hourly_variables: list[str],
    timezone: str = "Europe/Berlin",
) -> dict:
    """Call Open-Meteo Historical Weather Archive API."""
    params = {
        "latitude": latitude,
        "longitude": longitude,
        "start_date": start_date,
        "end_date": end_date,
        "hourly": ",".join(hourly_variables),
        "timezone": timezone,
    }
    response = requests.get(
        OPEN_METEO_ARCHIVE_URL,
        params=params,
        timeout=REQUEST_TIMEOUT,
    )
    response.raise_for_status()
    return response.json()


print(f"Fetching weather: ({DEMO_LATITUDE}, {DEMO_LONGITUDE})")
print(f"Date range: {start_date} to {end_date}")
print("(Requires internet — may take a few seconds)")
print()

api_response = fetch_open_meteo_archive(
    latitude=DEMO_LATITUDE,
    longitude=DEMO_LONGITUDE,
    start_date=start_date,
    end_date=end_date,
    hourly_variables=HOURLY_VARIABLES,
    timezone=DEMO_TIMEZONE,
)

# =============================================================================
# 4. PARSE WEATHER RESPONSE → DATAFRAME
# =============================================================================
def parse_hourly_response(response: dict) -> pd.DataFrame:
    """Convert Open-Meteo hourly JSON to flat DataFrame."""
    hourly = response["hourly"]
    df = pd.DataFrame(hourly)
    df["time"] = pd.to_datetime(df["time"])
    df["latitude"] = response.get("latitude")
    df["longitude"] = response.get("longitude")
    df["timezone"] = response.get("timezone")
    return df


weather_df = parse_hourly_response(api_response)

# Verify all requested variables present
missing_vars = [v for v in HOURLY_VARIABLES if v not in weather_df.columns]
if missing_vars:
    raise ValueError(f"API response missing variables: {missing_vars}")

# Save weather to disk (Notebook 04 loads/extends this pattern per station)
weather_df.to_parquet(WEATHER_PARQUET, index=False)

# =============================================================================
# 5. TIME ALIGNMENT — create merge key on railway side
# =============================================================================
def floor_to_hour(dt_series: pd.Series, tz: str = "Europe/Berlin") -> pd.Series:
    """
    Floor timestamps to the hour — this becomes the railway merge key.

    Example: 2024-07-01 14:37:00 → 2024-07-01 14:00:00
    """
    # Localize to Europe/Berlin if naive; then floor
    if dt_series.dt.tz is None:
        localized = dt_series.dt.tz_localize(tz, ambiguous="NaT", nonexistent="NaT")
    else:
        localized = dt_series.dt.tz_convert(tz)
    return localized.dt.floor("h").dt.tz_localize(None)  # naive for join comparison


# Apply to sample ICE rows that have departure_planned_time
ice_with_dep = ice_df[ice_df["departure_planned_time"].notna()].copy()
ice_with_dep["departure_planned_hour"] = floor_to_hour(
    ice_with_dep["departure_planned_time"]
)

# Sample 10 rows for alignment demonstration
alignment_sample = ice_with_dep[
    ["eva", "xml_station_name", "departure_planned_time",
     "departure_planned_hour", "delay_in_min"]
].head(10).copy()

# Weather side: time column IS the hour key
weather_df["weather_hour"] = weather_df["time"]

# =============================================================================
# 6. FORMAT COMPARISON
# =============================================================================
time_format_comparison = {
    "railway_raw": {
        "column": "departure_planned_time",
        "example": str(alignment_sample["departure_planned_time"].iloc[0]),
        "precision": "minute-level (or second)",
        "timezone": "naive in raw parquet — localized to Europe/Berlin in Notebook 03",
    },
    "railway_merge_key": {
        "column": "departure_planned_hour",
        "example": str(alignment_sample["departure_planned_hour"].iloc[0]),
        "precision": "hour-level (floored)",
        "timezone": "Europe/Berlin",
    },
    "weather_merge_key": {
        "column": "hourly.time",
        "example": str(weather_df["time"].iloc[0]),
        "precision": "hour-level",
        "timezone": api_response.get("timezone", "Europe/Berlin"),
    },
    "merge_rule": (
        "Join on: latitude + longitude + departure_planned_hour == weather.time"
    ),
}

# Check how many unique railway hours exist in ICE data
unique_railway_hours = ice_with_dep["departure_planned_hour"].nunique()
unique_weather_hours = weather_df["time"].nunique()

# Weather summary stats for July
weather_summary = {
    "rows": len(weather_df),
    "expected_hours": (
        (pd.Timestamp(end_date) - pd.Timestamp(start_date)).days + 1
    ) * 24,
    "date_min": str(weather_df["time"].min()),
    "date_max": str(weather_df["time"].max()),
    "temperature_2m_mean": round(float(weather_df["temperature_2m"].mean()), 2),
    "temperature_2m_min": round(float(weather_df["temperature_2m"].min()), 2),
    "temperature_2m_max": round(float(weather_df["temperature_2m"].max()), 2),
    "precipitation_total_mm": round(float(weather_df["precipitation"].sum()), 2),
    "hours_with_rain": int((weather_df["precipitation"] > 0).sum()),
}

# =============================================================================
# 7. SAVE INSPECTION REPORT
# =============================================================================
inspection_report = {
    "metadata": {
        "notebook": "02_Data_Acquisition_and_Initial_Inspection",
        "section": "Section 3",
        "created_at_utc": datetime.now(timezone.utc).isoformat(),
        "target_month": TARGET_MONTH,
        "ice_parquet": str(ICE_PARQUET),
        "weather_parquet": str(WEATHER_PARQUET),
    },
    "api_request": {
        "url": OPEN_METEO_ARCHIVE_URL,
        "latitude": DEMO_LATITUDE,
        "longitude": DEMO_LONGITUDE,
        "start_date": start_date,
        "end_date": end_date,
        "timezone": DEMO_TIMEZONE,
        "hourly_variables": HOURLY_VARIABLES,
        "note": "Demo location only — all eva stations geocoded in Notebook 04",
    },
    "api_response_meta": {
        "returned_latitude": api_response.get("latitude"),
        "returned_longitude": api_response.get("longitude"),
        "timezone": api_response.get("timezone"),
        "hourly_units": api_response.get("hourly_units", {}),
    },
    "time_format_comparison": time_format_comparison,
    "alignment_stats": {
        "ice_rows_with_departure": int(len(ice_with_dep)),
        "unique_railway_hours": int(unique_railway_hours),
        "unique_weather_hours": int(unique_weather_hours),
        "unique_eva_stations": int(ice_df["eva"].nunique()),
    },
    "weather_summary": weather_summary,
    "alignment_sample": alignment_sample.astype(str).to_dict(orient="records"),
    "next_steps": [
        "Notebook 03: standardize all timestamps to Europe/Berlin 24h format",
        "Notebook 04: geocode all eva stations; fetch weather per station",
        "Notebook 04: left join weather onto ICE rows using merge keys",
    ],
}

with WEATHER_REPORT_PATH.open("w", encoding="utf-8") as f:
    json.dump(inspection_report, f, indent=2, ensure_ascii=False)

# =============================================================================
# 8. PRINT SUMMARY
# =============================================================================
sep = "=" * 72
print(sep)
print("Section 3: Open-Meteo Weather Fetch & Time Alignment Check")
print(sep)
print(f"ICE date range     : {start_date} → {end_date}")
print(f"Weather rows saved : {len(weather_df):,}")
print(f"Expected hours     : {weather_summary['expected_hours']}")
print()

print("--- TIME FORMAT COMPARISON ---")
for side, info in time_format_comparison.items():
    if side == "merge_rule":
        print(f"  Rule: {info}")
    else:
        print(f"  [{side}]")
        print(f"    column   : {info['column']}")
        print(f"    example  : {info['example']}")
        print(f"    precision: {info['precision']}")
print()

print("--- WEATHER SUMMARY (Berlin demo, July 2024) ---")
for key, val in weather_summary.items():
    print(f"  {key}: {val}")
print()

print("--- ALIGNMENT STATS ---")
print(f"  ICE rows with departure time : {len(ice_with_dep):,}")
print(f"  Unique railway hours         : {unique_railway_hours:,}")
print(f"  Unique weather hours         : {unique_weather_hours:,}")
print(f"  Unique eva stations (ICE)    : {ice_df['eva'].nunique()}")
print()

print("--- HOUR ROUNDING DEMO (first 5 ICE rows) ---")
print(
    alignment_sample.head(5).to_string(index=False)
)
print()

print("--- SAVED ---")
print(f"  Weather parquet : {WEATHER_PARQUET}")
print(f"  Inspection report: {WEATHER_REPORT_PATH}")
print()
print("Ready for Section 4: Notebook 02 acquisition summary.")
print(sep)

ICE date range: 2024-07-01 to 2024-07-31
ICE rows      : 157,886
Unique eva    : 95  (geocoding in Notebook 04)

Fetching weather: (52.525, 13.369)
Date range: 2024-07-01 to 2024-07-31
(Requires internet — may take a few seconds)

Section 3: Open-Meteo Weather Fetch & Time Alignment Check
ICE date range     : 2024-07-01 → 2024-07-31
Weather rows saved : 744
Expected hours     : 744

--- TIME FORMAT COMPARISON ---
  [railway_raw]
    column   : departure_planned_time
    example  : 2024-07-01 00:01:00
    precision: minute-level (or second)
  [railway_merge_key]
    column   : departure_planned_hour
    example  : 2024-07-01 00:00:00
    precision: hour-level (floored)
  [weather_merge_key]
    column   : hourly.time
    example  : 2024-07-01 00:00:00
    precision: hour-level
  Rule: Join on: latitude + longitude + departure_planned_hour == weather.time

--- WEATHER SUMMARY (Berlin demo, July 2024) ---
  rows: 744
  expected_hours: 744
  date_min: 2024-07-01 00:00:00
  date_max: 2024-0

# Section 4: Acquisition Summary & Notebook 02 Close-Out

## Objective

Verify all acquired files exist on disk, summarize what was loaded in Notebook 02, and confirm readiness for Notebook 03 (cleaning).

## Why this step is important

A structured close-out proves the acquisition phase is complete and reproducible — any later notebook can reload from saved files without repeating downloads.

## What problem does it solve?

It answers: *What data do we now have on disk, and is it enough to start cleaning?*

## Methodology

Load all JSON reports and parquet files from `data/processed/` and `data/reference/`. Cross-check row counts, date ranges, and file presence. Save `notebook_02_summary.json`.

## Expected Output

- Checklist of all Notebook 02 artifacts (OK / MISSING)
- Summary table: ICE rows, stations, weather hours, date range
- `notebook_02_summary.json` saved
- Ready for Notebook 03: YES

## Interpretation

- **157k ICE rows, 95 stations** → solid sample for one month
- **744 weather hours** → full July 2024 hourly coverage for demo location
- **All files OK** → Notebook 03 loads `ice_raw_2024-07.parquet` directly
- **Geocoding pending** → 95 `eva` stations need coordinates in Notebook 04

## Challenges Faced

| Challenge | Resolution |
|-----------|------------|
| 54.8 GB full dataset | Loaded 1 month, ICE-filtered only |
| Path mismatch | `find_project_root()` in all sections |
| Weather only for Berlin demo | Full per-station fetch in Notebook 04 |
| Class imbalance | Documented in profile report |

## Summary

Notebook 02 complete. ICE operational data and weather sample are acquired, profiled, and saved to disk.

## Next Step

**Notebook 03, Section 1:** Standardize all timestamps to 24-hour `Europe/Berlin` format and apply cleaning rules.

---

### Notebook 02 — Closing

**Key Takeaways**
- **157,886 ICE stops** acquired for July 2024 across **95 stations**
- Saved: `ice_raw_2024-07.parquet` + profile/load/weather reports
- Hour-rounding merge key verified: `departure_planned_time` → `departure_planned_hour`
- Weather demo: 744 hourly rows for Berlin; full station coverage in Notebook 04

**What Comes Next**
- Notebook 03: timestamp standardization, handle nulls, exclude cancellations

In [5]:
# =============================================================================
# Notebook 02 | Section 4: Acquisition Summary & Notebook 02 Close-Out
# =============================================================================
# Self-contained: loads all Notebook 02 artifacts from disk and verifies them.
# Saves notebook_02_summary.json — confirms readiness for Notebook 03.
# =============================================================================

from __future__ import annotations

import json
from datetime import datetime, timezone
from pathlib import Path

import pandas as pd

# =============================================================================
# 1. PATHS
# =============================================================================
def find_project_root() -> Path:
    for candidate in [Path.cwd(), *Path.cwd().parents]:
        config_file = candidate / "data" / "reference" / "project_config.json"
        if config_file.exists():
            return candidate
    raise FileNotFoundError("Cannot find data/reference/project_config.json")


PROJECT_ROOT = find_project_root()
REFERENCE_DIR = PROJECT_ROOT / "data" / "reference"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"

TARGET_MONTH = "2024-07"
SUMMARY_PATH = REFERENCE_DIR / "notebook_02_summary.json"

# All artifacts expected from Notebook 02
ARTIFACTS = {
    "ice_raw_parquet": PROCESSED_DIR / f"ice_raw_{TARGET_MONTH}.parquet",
    "weather_sample_parquet": PROCESSED_DIR / f"weather_sample_{TARGET_MONTH}.parquet",
    "ice_load_report": REFERENCE_DIR / f"ice_load_report_{TARGET_MONTH}.json",
    "ice_profile_report": REFERENCE_DIR / f"ice_profile_report_{TARGET_MONTH}.json",
    "weather_inspection_report": REFERENCE_DIR / f"weather_inspection_report_{TARGET_MONTH}.json",
}

# Prior notebook artifacts still required
PRIOR_ARTIFACTS = {
    "project_config": REFERENCE_DIR / "project_config.json",
    "research_framework": REFERENCE_DIR / "research_framework.json",
    "db_data_dictionary": REFERENCE_DIR / "db_data_dictionary.json",
    "weather_data_dictionary": REFERENCE_DIR / "weather_data_dictionary.json",
    "merge_strategy": REFERENCE_DIR / "merge_strategy.json",
}


def load_json(path: Path) -> dict:
    if not path.exists():
        raise FileNotFoundError(f"Missing: {path}")
    with path.open("r", encoding="utf-8") as f:
        return json.load(f)


# =============================================================================
# 2. VERIFY ALL FILES ON DISK
# =============================================================================
def check_file(path: Path) -> dict:
    """Return file status and size."""
    if path.exists():
        size_mb = round(path.stat().st_size / (1024 * 1024), 2)
        return {"status": "OK", "path": str(path), "size_mb": size_mb}
    return {"status": "MISSING", "path": str(path), "size_mb": None}


print("--- NOTEBOOK 02 ARTIFACT CHECK ---")
artifact_status = {name: check_file(path) for name, path in ARTIFACTS.items()}
for name, info in artifact_status.items():
    size_str = f"({info['size_mb']} MB)" if info["size_mb"] is not None else ""
    print(f"  [{info['status']}] {name} {size_str}")

missing_nb02 = [n for n, i in artifact_status.items() if i["status"] == "MISSING"]
if missing_nb02:
    raise FileNotFoundError(
        f"Missing Notebook 02 files: {missing_nb02}\n"
        "Complete Sections 1–3 before running this cell."
    )

print()
print("--- NOTEBOOK 01 ARTIFACT CHECK ---")
prior_status = {name: check_file(path) for name, path in PRIOR_ARTIFACTS.items()}
for name, info in prior_status.items():
    print(f"  [{info['status']}] {name}")

missing_prior = [n for n, i in prior_status.items() if i["status"] == "MISSING"]
if missing_prior:
    print(f"  WARNING: Missing Notebook 01 files: {missing_prior}")

# =============================================================================
# 3. LOAD REPORTS & BUILD SUMMARY
# =============================================================================
load_report = load_json(ARTIFACTS["ice_load_report"])
profile_report = load_json(ARTIFACTS["ice_profile_report"])
weather_report = load_json(ARTIFACTS["weather_inspection_report"])

# Quick reload of parquet shapes (confirms files are readable)
ice_df = pd.read_parquet(ARTIFACTS["ice_raw_parquet"])
weather_df = pd.read_parquet(ARTIFACTS["weather_sample_parquet"])

acquisition_summary = {
    "metadata": {
        "notebook": "02_Data_Acquisition_and_Initial_Inspection",
        "section": "Section 4 — Close-Out",
        "created_at_utc": datetime.now(timezone.utc).isoformat(),
        "target_month": TARGET_MONTH,
        "status": "COMPLETE",
    },
    "ice_data": {
        "rows": int(len(ice_df)),
        "columns": int(len(ice_df.columns)),
        "unique_eva_stations": profile_report["station_summary"]["unique_eva_stations"],
        "unique_train_numbers": profile_report["train_summary"]["unique_train_numbers"],
        "unique_rides": profile_report["train_summary"]["unique_rides"],
        "canceled_pct": profile_report["cancellation"]["canceled_pct"],
        "date_range": profile_report["time_range"],
        "delay_mean_min": profile_report["delay_distribution"]["mean"],
        "delayed_gte_6_pct": round(
            100 * profile_report["delay_distribution"]["late_gte_6"]
            / profile_report["delay_distribution"]["count"],
            2,
        ),
        "source_month_scanned": load_report["processing"]["total_rows_scanned"],
        "ice_pct_of_month": load_report["processing"]["ice_pct_of_month"],
        "parquet_file": str(ARTIFACTS["ice_raw_parquet"]),
    },
    "weather_data": {
        "rows": int(len(weather_df)),
        "date_range": {
            "min": str(weather_df["time"].min()),
            "max": str(weather_df["time"].max()),
        },
        "demo_location": weather_report["api_request"],
        "temperature_mean_c": weather_report["weather_summary"]["temperature_2m_mean"],
        "precipitation_total_mm": weather_report["weather_summary"]["precipitation_total_mm"],
        "parquet_file": str(ARTIFACTS["weather_sample_parquet"]),
        "note": "Demo Berlin location only — all 95 eva stations in Notebook 04",
    },
    "time_alignment": {
        "railway_key": "departure_planned_hour (floored from departure_planned_time)",
        "weather_key": "hourly.time",
        "timezone": "Europe/Berlin",
        "unique_railway_hours": weather_report["alignment_stats"]["unique_railway_hours"],
        "unique_weather_hours": weather_report["alignment_stats"]["unique_weather_hours"],
        "verified": True,
    },
    "quality_flags": profile_report["quality_flags"],
    "artifacts": {
        name: info for name, info in artifact_status.items()
    },
    "ready_for_notebook_03": True,
    "notebook_03_tasks": [
        "Standardize all timestamps to Europe/Berlin 24h format",
        "Exclude is_canceled == True rows",
        "Handle null values in station_name and line_number",
        "Save ice_cleaned_2024-07.parquet",
    ],
}

with SUMMARY_PATH.open("w", encoding="utf-8") as f:
    json.dump(acquisition_summary, f, indent=2, ensure_ascii=False)

# =============================================================================
# 4. PRINT NOTEBOOK 02 SUMMARY
# =============================================================================
sep = "=" * 72
print()
print(sep)
print("Notebook 02 — Acquisition Summary")
print(sep)

ice = acquisition_summary["ice_data"]
wx = acquisition_summary["weather_data"]
align = acquisition_summary["time_alignment"]

print()
print("--- ICE DATA (July 2024) ---")
print(f"  Rows              : {ice['rows']:,}")
print(f"  Stations (eva)    : {ice['unique_eva_stations']}")
print(f"  Train services    : {ice['unique_train_numbers']}")
print(f"  Canceled stops    : {ice['canceled_pct']}%")
print(f"  Mean delay        : {ice['delay_mean_min']} min")
print(f"  Delayed ≥6 min    : {ice['delayed_gte_6_pct']}%")
print(f"  ICE % of month    : {ice['ice_pct_of_month']}%")
print()

print("--- WEATHER DATA (demo: Berlin) ---")
print(f"  Hourly rows       : {wx['rows']:,}")
print(f"  Mean temperature  : {wx['temperature_mean_c']} °C")
print(f"  Total precipitation: {wx['precipitation_total_mm']} mm")
print()

print("--- TIME ALIGNMENT ---")
print(f"  Railway key  : {align['railway_key']}")
print(f"  Weather key  : {align['weather_key']}")
print(f"  Verified     : {align['verified']}")
print()

print("--- SAVED FILES ---")
for name, info in artifact_status.items():
    print(f"  {info['path']}  ({info['size_mb']} MB)")
print(f"  {SUMMARY_PATH}")
print()

print("=" * 72)
print("NOTEBOOK 02 COMPLETE — Ready for Notebook 03")
print("=" * 72)
print()
print("Key Takeaways:")
print(f"  • {ice['rows']:,} ICE stops | {ice['unique_eva_stations']} stations | July 2024")
print(f"  • {wx['rows']} hourly weather rows (Berlin demo)")
print("  • Hour-rounding merge key verified")
print("  • All data saved to disk — fully reproducible")
print()
print("What Comes Next:")
print("  → Notebook 03, Section 1: Timestamp standardization & cleaning")
print(sep)

--- NOTEBOOK 02 ARTIFACT CHECK ---
  [OK] ice_raw_parquet (5.24 MB)
  [OK] weather_sample_parquet (0.02 MB)
  [OK] ice_load_report (0.0 MB)
  [OK] ice_profile_report (0.01 MB)
  [OK] weather_inspection_report (0.0 MB)

--- NOTEBOOK 01 ARTIFACT CHECK ---
  [OK] project_config
  [OK] research_framework
  [OK] db_data_dictionary
  [OK] weather_data_dictionary
  [OK] merge_strategy

Notebook 02 — Acquisition Summary

--- ICE DATA (July 2024) ---
  Rows              : 157,886
  Stations (eva)    : 95
  Train services    : 812
  Canceled stops    : 7.28%
  Mean delay        : 10.9 min
  Delayed ≥6 min    : 41.89%
  ICE % of month    : 7.87%

--- WEATHER DATA (demo: Berlin) ---
  Hourly rows       : 744
  Mean temperature  : 20.37 °C
  Total precipitation: 58.7 mm

--- TIME ALIGNMENT ---
  Railway key  : departure_planned_hour (floored from departure_planned_time)
  Weather key  : hourly.time
  Verified     : True

--- SAVED FILES ---
  c:\Users\Manikanta\Desktop\ML Project\Notebooks\data\pro